In [0]:
# =========================================================
# Copy CSVs from Workspace -> Unity Catalog Volume
# =========================================================

WORKSPACE_DATA_DIR = "dbfs:/Workspace/Projects/Capstone/data"
VOLUME_SELECTED_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw"

dbutils.fs.mkdirs(VOLUME_SELECTED_DIR)

workspace_files = dbutils.fs.ls(WORKSPACE_DATA_DIR)

csv_files = [f for f in workspace_files if f.path.lower().endswith(".csv")]

print(f"Found {len(csv_files)} CSV files in Workspace")

for f in csv_files:
    filename = f.path.split("/")[-1]
    dest = f"{VOLUME_SELECTED_DIR}/{filename}"
    print(f"Copying {filename}")
    dbutils.fs.cp(f.path, dest, True)

print("\n=== Copy complete ===")
display(dbutils.fs.ls(VOLUME_SELECTED_DIR))


Found 24 CSV files in Workspace
Copying Bike share ridership 2022-10.csv
Copying Bike share ridership 2022-11.csv
Copying Bike share ridership 2022-12.csv
Copying Bike share ridership 2023-01.csv
Copying Bike share ridership 2023-02.csv
Copying Bike share ridership 2023-03.csv
Copying Bike share ridership 2023-04.csv
Copying Bike share ridership 2023-05.csv
Copying Bike share ridership 2023-06.csv
Copying Bike share ridership 2023-07.csv
Copying Bike share ridership 2023-08.csv
Copying Bike share ridership 2023-09.csv
Copying Bike share ridership 2023-10.csv
Copying Bike share ridership 2023-11.csv
Copying Bike share ridership 2023-12.csv
Copying Bike share ridership 2024-01.csv
Copying Bike share ridership 2024-02.csv
Copying Bike share ridership 2024-03.csv
Copying Bike share ridership 2024-04.csv
Copying Bike share ridership 2024-05.csv
Copying Bike share ridership 2024-06.csv
Copying Bike share ridership 2024-07.csv
Copying Bike share ridership 2024-08.csv
Copying Bike share riders

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2022-10.csv,Bike share ridership 2022-10.csv,64985321,1770075071000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2022-11.csv,Bike share ridership 2022-11.csv,41474040,1770075074000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2022-12.csv,Bike share ridership 2022-12.csv,23274638,1770075076000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-01.csv,Bike share ridership 2023-01.csv,23327418,1770075078000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-02.csv,Bike share ridership 2023-02.csv,22374657,1770075080000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-03.csv,Bike share ridership 2023-03.csv,28990070,1770075081000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-04.csv,Bike share ridership 2023-04.csv,49001380,1770075082000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-05.csv,Bike share ridership 2023-05.csv,75905850,1770075085000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-06.csv,Bike share ridership 2023-06.csv,84923348,1770075087000
dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw/Bike share ridership 2023-07.csv,Bike share ridership 2023-07.csv,94068141,1770075090000


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
SELECTED_RAW_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw"
BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership"

dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze")

print("SELECTED_RAW_DIR:", SELECTED_RAW_DIR)
print("BRONZE_DIR      :", BRONZE_DIR)

# ---------------------------------------------------------
# 1) Read CSVs (Unity Catalog Volume)
# ---------------------------------------------------------
print("\nReading CSV files...")
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv(SELECTED_RAW_DIR))

# ---------------------------------------------------------
# 2) Get file path using Unity Catalog supported metadata
#    (instead of input_file_name())
# ---------------------------------------------------------
df = df.withColumn("source_file", F.col("_metadata.file_path"))

# Extract year and month from filename (YYYY-MM)
df = df.withColumn("year", F.regexp_extract("source_file", r"(20\d{2})-(\d{2})", 1).cast("int"))
df = df.withColumn("month", F.regexp_extract("source_file", r"(20\d{2})-(\d{2})", 2).cast("int"))

# ---------------------------------------------------------
# 3) Safety filter: Oct-2022 -> Sep-2024
# ---------------------------------------------------------
df = df.filter(
    ((F.col("year") == 2022) & (F.col("month").between(10, 12))) |
    ((F.col("year") == 2023) & (F.col("month").between(1, 12))) |
    ((F.col("year") == 2024) & (F.col("month").between(1, 9)))
)

# Optional: Check for any rows that failed extraction
bad = df.filter(F.col("year").isNull() | F.col("month").isNull()).count()
if bad > 0:
    print(f"Rows with missing year/month: {bad} (check filename format)")
else:
    print("year/month extracted successfully from _metadata.file_path")

total_rows = df.count()
print(f"\nTotal rows (Oct-2022..Sep-2024): {total_rows:,}")

# ---------------------------------------------------------
# 4) Write Bronze Parquet partitioned by year/month
# ---------------------------------------------------------
print("\nWriting Bronze Parquet (partitionBy year, month)...")

# Clean previous bronze output (optional)
try:
    dbutils.fs.rm(BRONZE_DIR, True)
except Exception:
    pass

(df.write
 .mode("overwrite")
 .partitionBy("year", "month")
 .parquet(BRONZE_DIR))

print("Bronze Parquet written to:", BRONZE_DIR)

# ---------------------------------------------------------
# 5) Validation: counts per month + expected 24 months
# ---------------------------------------------------------
print("\nPost-write validation...")

bronze_df = spark.read.parquet(BRONZE_DIR)

# Total rows from parquet
bronze_total = bronze_df.count()
print(f"Total rows in Bronze Parquet: {bronze_total:,}")

# Count rows per year
print("\n=== Records per year ===")
(bronze_df
 .groupBy("year")
 .agg(F.count("*").alias("rows"))
 .orderBy("year")
 .show(truncate=False))

# Count rows per year & month
print("\n=== Records per year & month ===")
month_counts = (bronze_df
                .groupBy("year", "month")
                .agg(F.count("*").alias("rows"))
                .orderBy("year", "month"))
month_counts.show(50, truncate=False)

# Distinct month partitions
distinct_months = bronze_df.select("year", "month").distinct().count()
print(f"\nDistinct (year, month) partitions: {distinct_months}")

if distinct_months == 24:
    print("Month coverage OK (24 months)")
else:
    print("Month coverage NOT OK (expected 24). Check missing files.")

print("\n=== DONE ===")
print("Downstream read example:")
print(f'df = spark.read.parquet("{BRONZE_DIR}")')


SELECTED_RAW_DIR: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/selected_raw
BRONZE_DIR      : dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership

Reading CSV files...
✅ year/month extracted successfully from _metadata.file_path

Total rows (Oct-2022..Sep-2024): 12,055,519

Writing Bronze Parquet (partitionBy year, month)...
✅ Bronze Parquet written to: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership

Post-write validation...
Total rows in Bronze Parquet: 12,055,519

=== Records per year ===
+----+-------+
|year|rows   |
+----+-------+
|2022|999990 |
|2023|5713141|
|2024|5342388|
+----+-------+


=== Records per year & month ===
+----+-----+------+
|year|month|rows  |
+----+-----+------+
|2022|10   |499751|
|2022|11   |320229|
|2022|12   |180010|
|2023|1    |180135|
|2023|2    |172776|
|2023|3    |224545|
|2023|4    |380346|
|2023|5    |589217|
|2023|6    |663691|
|2023|7    |735924|
|2023|8    |76

In [0]:
df.printSchema()

root
 |-- Trip Id: string (nullable = true)
 |-- Trip  Duration: string (nullable = true)
 |-- Start Station Id: string (nullable = true)
 |-- Start Time: string (nullable = true)
 |-- Start Station Name: string (nullable = true)
 |-- End Station Id: string (nullable = true)
 |-- End Time: string (nullable = true)
 |-- End Station Name: string (nullable = true)
 |-- Bike Id: string (nullable = true)
 |-- User Type: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

